In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import mlflow
import mlflow.sklearn


from src.data_preparation import clean_data
from src.feature_engineering import add_features
from src.preprocessing import build_preprocessor

In [2]:
df = pd.read_csv("../data/raw/House_Prices.csv")

In [3]:
df = clean_data(df)
df = add_features(df)

In [4]:
X = df.drop(columns="SalePrice")
y = df["SalePrice"]

In [5]:
preprocessor = build_preprocessor(X)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
gradient_boosting_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42))
])

In [ ]:
mlflow.set_tracking_uri(
    "sqlite:////home/elgmouri/projects/aqari/mlflow.db"
)
mlflow.set_experiment("Aqari Model Comparison")
with mlflow.start_run(run_name="Final Gradient Boosting"):
    gradient_boosting_pipeline.fit(X_train, y_train)
    y_pred = gradient_boosting_pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)


    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)

    mlflow.sklearn.log_model(
        gradient_boosting_pipeline,
        name="model",
        serialization_format="cloudpickle"
    )

2026/09/28 12:01:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [9]:
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 14531.808750105632
RMSE: 20072.89440595186
R²: 0.9270563038889157


In [10]:
results = pd.DataFrame({
    "ActualPrice": y_test,
    "PredictedPrice": y_pred
})

results["Error"] = results["ActualPrice"] - results["PredictedPrice"]
results["AbsoluteError"] = results["Error"].abs()

results.sort_values(
    "AbsoluteError",
    ascending=False
).head(10)

,ActualPrice,PredictedPrice,Error,AbsoluteError
218,311500.0,219770.666348,91729.333652,91729.333652
261,276000.0,362742.610351,-86742.610351,86742.610351
1044,278000.0,348375.273855,-70375.273855,70375.273855
666,129000.0,190380.849108,-61380.849108,61380.849108
332,284000.0,345225.913909,-61225.913909,61225.913909
1031,197000.0,255231.229584,-58231.229584,58231.229584
716,159500.0,215981.484962,-56481.484962,56481.484962
1415,175900.0,223268.236701,-47368.236701,47368.236701
1310,335000.0,287636.757019,47363.242981,47363.242981
56,172500.0,218645.985284,-46145.985284,46145.985284


The original Gradient Boosting model was selected as the final model because it performed better than the tuned version on the holdout test set.

The final model was evaluated using MAE, RMSE and R², and the largest prediction errors were analyzed.